# Adjudication of sepsis patients in EARLI cohort with reasoning models

Here I process validation cohort with GPT-5.2

Kernel: `.venv (Python 3.13.5)` via `~/venvs/20240903_LLM_sepsis_prediction`

## Set up packages and API access

In [1]:
import os
from pathlib import Path
# import json
import time

import numpy as np
import polars as pl

import matplotlib.pyplot as plt
import seaborn as sns

from dotenv import load_dotenv
from openai import AzureOpenAI

In [2]:
load_dotenv(".env")

API_KEY = os.getenv("API_KEY")
API_VERSION = "2025-04-01-preview"  # latest preview API release
RESOURCE_ENDPOINT = os.getenv("AZURE_ENDPOINT")

client = AzureOpenAI(
    api_key=API_KEY,
    api_version=API_VERSION,
    azure_endpoint=RESOURCE_ENDPOINT,
)
client

Set up paths

In [12]:
project_path = "/Users/hoangvanphan/Library/CloudStorage/Box-Box/Van_research/20240903_LLM_sepsis_prediction"
note_path_ucsf = "/Users/hoangvanphan/Library/CloudStorage/Box-Box/Sepsis_GPT4/DerivationCohort/^Secure_ER_Notes/ER_notes_txt"
note_path_zsfg = "/Users/hoangvanphan/Library/CloudStorage/Box-Box/Sepsis_GPT4/DerivationCohort/^Secure_ER_notes_ZFGH/ER_notes_txt"

## Import metadata

In [4]:
metadata = pl.read_csv(
    "/Users/hoangvanphan/Library/CloudStorage/Box-Box/Sepsis_GPT4/DerivationCohort/EARLI_Metadata/DerivationCohortMetadata032726.csv",
    infer_schema=False,
)

metadata = metadata.select("Barcode","EARLIStudyID","Sepsis","Hospital Death")
print(metadata.head())

shape: (5, 4)
┌─────────┬──────────────┬───────────────┬────────────────┐
│ Barcode ┆ EARLIStudyID ┆ Sepsis        ┆ Hospital Death │
│ ---     ┆ ---          ┆ ---           ┆ ---            │
│ str     ┆ str          ┆ str           ┆ str            │
╞═════════╪══════════════╪═══════════════╪════════════════╡
│ 10447   ┆ 284          ┆ SepsisCx-     ┆ No             │
│ 10456   ┆ 289          ┆ SepsisCx+     ┆ No             │
│ 10535   ┆ 340          ┆ Indeterminate ┆ Yes            │
│ 10584   ┆ 368          ┆ Indeterminate ┆ No             │
│ 10691   ┆ 437          ┆ SepsisCx-     ┆ No             │
└─────────┴──────────────┴───────────────┴────────────────┘


Replace `-` with `_` to be consistent with the note filenames.

In [5]:
metadata = (
    metadata
    .with_columns(pl.col("Barcode").str.replace_all("-", "_"))
    .sort("Barcode")
)
print(metadata.head())

shape: (5, 4)
┌─────────┬──────────────┬───────────────┬────────────────┐
│ Barcode ┆ EARLIStudyID ┆ Sepsis        ┆ Hospital Death │
│ ---     ┆ ---          ┆ ---           ┆ ---            │
│ str     ┆ str          ┆ str           ┆ str            │
╞═════════╪══════════════╪═══════════════╪════════════════╡
│ 10447   ┆ 284          ┆ SepsisCx-     ┆ No             │
│ 10456   ┆ 289          ┆ SepsisCx+     ┆ No             │
│ 10535   ┆ 340          ┆ Indeterminate ┆ Yes            │
│ 10584   ┆ 368          ┆ Indeterminate ┆ No             │
│ 10691   ┆ 437          ┆ SepsisCx-     ┆ No             │
└─────────┴──────────────┴───────────────┴────────────────┘


## Custom functions

API call

In [6]:
def reasoning_adjudication(
        note: str,
        system_prompt: str,
        model: str = "gpt-5-2025-08-07",
        n: int = 1,
        max_completion_tokens: int = 10000,
        reasoning_effort: str = "low",
        # temperature: float = 0.2,  # not used by reasoning models
        # seed: int = 0  # not used by reasoning models
    ):
    # Output: the API response

    messages = [
        {"role": "developer", "content": system_prompt},
        {"role": "user", "content": note}
    ]

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        n=n,
        max_completion_tokens=max_completion_tokens,
        reasoning_effort=reasoning_effort,
    )

    if response.choices[0].finish_reason != "stop":
        raise Exception(f"Finish reason is not 'stop'! It's {response.choices[0].finish_reason}")
    
    return response

## System prompt

In [7]:
mortality_prompt_20251106 = """
You are an AI clinician with expertise in critical care. You are trying to predict whether a critically ill patient is more or less likely to survive their hospitalization.
Below, you will be given the emergency room note for the patient. I want you to come to your own independent prediction of whether the patient is likely to survive to hospital discharge, regardless of what the clinical team thought. In your prediction, you should consider incorporating critical illness survival scores, such as APACHE-II, APACHE-III, or SAPS. However, you should also bear in mind that these calculators are not perfect. You also may not have all the information that you need to complete those calculators from the emergency room note. You are free to consider broadly the patient's clinical state, including co-morbidities, age, and current illness, and you should bear in mind that a majority of patients are expected to survive to discharge.
End your answer with a paragraph containing either ">>Survive" or ">>Not survive" to indicate your prediction, and nothing else.
"""

## GPT-5.2 adjudication

Remove the 13 training patients.

In [8]:
training_barcodes = [
    "10760",
    "10812",
    "11282",
    "11416",
    "11474",
    "11494",
    "11708",
    "11805",
    "11846",
    "11858",
    "11983",

    "50417",
    "50623",
]

In [9]:
print(metadata.shape)
metadata = metadata.filter(~pl.col("Barcode").is_in(training_barcodes))
print(metadata.shape)

(303, 4)
(290, 4)


In [10]:
print(metadata.shape)

(290, 4)


### Process the discovery cohort

10 adjudications per patient. Save results every 25 patients.

In [13]:
# Iterate
barcodes = metadata["Barcode"].to_list()
df_gpt52 = []

group_counter = 1
for counter, value in enumerate(barcodes[:50], start=1):
    note_path = note_path_ucsf if value[0] == "1" else note_path_zsfg

    print(f"Processing ID {value}")

    try:
        with open(Path(note_path, f"{value}.txt"), "r", errors="ignore", encoding="utf-8") as f1:
            note = f1.read()
    except FileNotFoundError:
        print(f"***** ID {value} not found *****")
        continue
    
    for k in range(2):
        response = reasoning_adjudication(
            note=note,
            system_prompt=mortality_prompt_20251106,
            model="gpt-5.2-2025-12-11",
            n=5,
            max_completion_tokens=10000,
            reasoning_effort="high",
        )

        for j in range(len(response.choices)):
            temp = {"Barcode": value, "run": k+1}
            temp.update(
                {"adjudication": response.choices[j].message.content,
                 "sepsis": response.choices[j].message.content.split(">>")[-1].strip()}
            )
            df_gpt52.append(temp)

        time.sleep(3)  # add sleep so that it doesn't overwhelm the requests per minute limit

    if ((counter % 25) == 0) or (counter == len(barcodes)):
        pl.DataFrame(df_gpt52).write_csv(
            Path(project_path, "Secure_out", f"20260407_GPT5.2_mortality_group{group_counter}.csv")
        )
        df_gpt52 = []
        group_counter = group_counter + 1

Processing ID 10447
Processing ID 10456
Processing ID 10535
Processing ID 10584
Processing ID 10691
Processing ID 10740
Processing ID 10741
Processing ID 10755
Processing ID 10757
Processing ID 10778
Processing ID 10819
Processing ID 10820
Processing ID 10825
Processing ID 10846
Processing ID 10856
Processing ID 10902
Processing ID 10929
Processing ID 10956
Processing ID 10957
Processing ID 10958
Processing ID 10963
Processing ID 10971
Processing ID 10979
Processing ID 10981
Processing ID 10983
Processing ID 10991
Processing ID 10993
Processing ID 11005
Processing ID 11010
Processing ID 11011
Processing ID 11017
Processing ID 11022
Processing ID 11024
Processing ID 11028
Processing ID 11035
Processing ID 11038
Processing ID 11043
Processing ID 11046
Processing ID 11052
Processing ID 11053
Processing ID 11062
Processing ID 11079
Processing ID 11080
Processing ID 11086
Processing ID 11093
Processing ID 11097
Processing ID 11111
Processing ID 11114
Processing ID 11122
Processing ID 11128


In [14]:
print(group_counter)

3


In [15]:
# Iterate
# barcodes = metadata["Barcode"].to_list()
df_gpt52 = []

group_counter = 3
for counter, value in enumerate(barcodes[50:], start=51):
    note_path = note_path_ucsf if value[0] == "1" else note_path_zsfg

    print(f"Processing ID {value}")

    try:
        with open(Path(note_path, f"{value}.txt"), "r", errors="ignore", encoding="utf-8") as f1:
            note = f1.read()
    except FileNotFoundError:
        print(f"***** ID {value} not found *****")
        continue
    
    for k in range(2):
        response = reasoning_adjudication(
            note=note,
            system_prompt=mortality_prompt_20251106,
            model="gpt-5.2-2025-12-11",
            n=5,
            max_completion_tokens=10000,
            reasoning_effort="high",
        )

        for j in range(len(response.choices)):
            temp = {"Barcode": value, "run": k+1}
            temp.update(
                {"adjudication": response.choices[j].message.content,
                 "sepsis": response.choices[j].message.content.split(">>")[-1].strip()}
            )
            df_gpt52.append(temp)

        time.sleep(3)  # add sleep so that it doesn't overwhelm the requests per minute limit

    if ((counter % 25) == 0) or (counter == len(barcodes)):
        pl.DataFrame(df_gpt52).write_csv(
            Path(project_path, "Secure_out", f"20260407_GPT5.2_mortality_group{group_counter}.csv")
        )
        df_gpt52 = []
        group_counter = group_counter + 1

Processing ID 11137
Processing ID 11139
Processing ID 11140
Processing ID 11164
Processing ID 11168
Processing ID 11175
Processing ID 11180
Processing ID 11182
Processing ID 11184
Processing ID 11195
Processing ID 11197
Processing ID 11203
Processing ID 11221
Processing ID 11254
Processing ID 11255
Processing ID 11284
Processing ID 11290
Processing ID 11297
Processing ID 11299
Processing ID 11300
Processing ID 11308
Processing ID 11309
***** ID 11309 not found *****
Processing ID 11310
Processing ID 11317
Processing ID 11320
Processing ID 11321
Processing ID 11324
Processing ID 11329
Processing ID 11343
Processing ID 11344
Processing ID 11345
Processing ID 11353
Processing ID 11362
Processing ID 11371
Processing ID 11373
Processing ID 11375
Processing ID 11379
Processing ID 11388
Processing ID 11403
Processing ID 11404
Processing ID 11411
Processing ID 11412
Processing ID 11417
Processing ID 11422
Processing ID 11424
Processing ID 11428
Processing ID 11430
Processing ID 11437
Processin

In [16]:
print(group_counter)

13


Process the encrypted notes

In [17]:
df_gpt52 = []
note_path = "/Users/hoangvanphan/Library/Containers/com.ciphercloud.macapp.CipherCloud/Data/Library/Caches/com.ciphercloud.macapp.CipherCloud/originalFile"

group_counter = 13
for value in ["11309","50372"]:
    print(f"Processing ID {value}")

    try:
        with open(Path(note_path, f"{value}.txt"), "r", errors="ignore", encoding="utf-8") as f1:
            note = f1.read()
    except FileNotFoundError:
        print(f"***** ID {value} not found *****")
        continue
    
    for k in range(2):
        response = reasoning_adjudication(
            note=note,
            system_prompt=mortality_prompt_20251106,
            model="gpt-5.2-2025-12-11",
            n=5,
            max_completion_tokens=10000,
            reasoning_effort="high",
        )

        for j in range(len(response.choices)):
            temp = {"Barcode": value, "run": k+1}
            temp.update(
                {"adjudication": response.choices[j].message.content,
                 "sepsis": response.choices[j].message.content.split(">>")[-1].strip()}
            )
            df_gpt52.append(temp)

        time.sleep(3)  # add sleep so that it doesn't overwhelm the requests per minute limit

pl.DataFrame(df_gpt52).write_csv(
    Path(project_path, "Secure_out", f"20260407_GPT5.2_mortality_group{group_counter}.csv")
)

Processing ID 11309
Processing ID 50372
